# Solution: Pareto Optimization of SEI Additives with ALCHEMI

## Control Panel

These settings keep the example small while preserving the Part 1 adsorption-search pattern: relaxed clean slabs, a compact site/orientation grid, batched relaxations, and lowest-energy valid-start selection.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.005
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05
TOOLKIT_FIRE2_MAXSTEP = 0.04
TOOLKIT_D3BJ = None
BATCH_SIZE = 2

SOLUTION_SETTINGS = {
    "min_adsorption_clearance_A": 1.6,
    "vdw_height_scale": 0.66,
    "surface_height_tolerance_A": 1.2,
    "gas_box_A": 20.0,
    "adsorption_site_limit": 3,
    "adsorption_azimuth_angles_deg": (0.0,),
    "max_surface_displacement_A": 1.5,
    "frozen_surface_fraction": 0.5,
}

EXAMPLE_SYSTEMS = (
    ("FEC", "li_metal"),
    ("FEC", "passivating"),
    ("VC", "li_metal"),
    ("VC", "passivating"),
    ("EMC", "li_metal"),
    ("EMC", "passivating"),
)

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The solution reuses the Part 1 Toolkit backend and keeps challenge-specific geometry/search utilities in `challenge_utils.solution_helpers`.


In [13]:
import os
import sys
import importlib
from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

os.environ["ALCHEMI_ALLOW_CACHE_OVERWRITE"] = "1"

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    display_widgets_grid,
)
from challenge_utils.molecules import build_molecule, known_molecules
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score
import challenge_utils.solution_helpers as solution_helpers

importlib.reload(solution_helpers)
from challenge_utils.solution_helpers import (
    SolutionSettings,
    binding_energy_table,
    build_adsorption_surfaces,
    choose_inspection_candidates,
    component_energy_table,
    inspection_widget_rows,
    make_clean_surface_jobs,
    make_combined_jobs,
    make_gas_jobs,
    prepare_challenge_tables,
    relax_structures,
    require_all_converged,
    select_lowest_energy_site_results,
    selected_site_summary,
    surface_summary,
    write_ovito_inspection_structures,
)

SETTINGS = SolutionSettings(**SOLUTION_SETTINGS)

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


Challenge folder : challenge-sei
Part 1 helpers   : part-1-batched-adsorption
ase               : 3.28.0
numpy             : 2.4.6
pandas            : 3.0.3
torch             : 2.12.0+cu130
nvalchemi-toolkit : 0.1.0
ovito             : 3.15.4


## 1. Load The Challenge Manifests

All molecule geometries are built **in code** (`challenge_utils.molecules`) — the challenge reads no structure files. Extend the starter panel with `data/custom_molecule_manifest.csv` plus a `custom_molecules.py` that registers each new geometry as an `ase.Atoms` (copy `custom_molecules_template.py`). The active run is reduced by `EXAMPLE_SYSTEMS` for quick iteration.


In [14]:
molecules_df = pd.read_csv("data/molecule_manifest.csv")
custom_manifest_path = Path("data/custom_molecule_manifest.csv")
if custom_manifest_path.exists():
    custom_molecules_df = pd.read_csv(custom_manifest_path)
    molecules_df = pd.concat([molecules_df, custom_molecules_df], ignore_index=True)
    print(f"Loaded {len(custom_molecules_df)} custom/literature molecule row(s).")
else:
    print("No data/custom_molecule_manifest.csv found; using the starter molecule panel.")

# Literature geometries are registered in code (no structure files): importing
# custom_molecules.py runs its register_molecule(...) calls.
if Path("custom_molecules.py").exists():
    import custom_molecules  # noqa: F401  (side effect: registers ase.Atoms geometries)
    print("Imported custom_molecules.py (registered literature geometries).")

if molecules_df["candidate_id"].duplicated().any():
    duplicated = sorted(molecules_df.loc[molecules_df["candidate_id"].duplicated(), "candidate_id"].unique())
    raise RuntimeError(f"Duplicate candidate_id value(s): {duplicated}")

surfaces_df = pd.read_csv("data/surface_manifest.csv")
lookup_df = pd.read_csv("data/class_surface_lookup.csv")
challenge_df, run_systems_df, run_challenge_df = prepare_challenge_tables(
    molecules_df,
    lookup_df,
    EXAMPLE_SYSTEMS,
)

unknown_geometries = sorted(set(challenge_df["candidate_id"]) - set(known_molecules()))
if unknown_geometries:
    raise RuntimeError(
        f"No in-code geometry for: {unknown_geometries}. Register each one in "
        "custom_molecules.py via challenge_utils.molecules.register_molecule "
        "(see custom_molecules_template.py)."
    )

assert set(challenge_df["role"]) == {"baseline", "additive"}
assert {"EC", "EMC"}.issubset(set(challenge_df.loc[challenge_df["role"].eq("baseline"), "candidate_id"]))

print(f"Run systems: {len(run_systems_df)} adsorption system(s); {len(run_challenge_df)} molecule reference(s).")
display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id", "formula"]])
display(surfaces_df[["surface_id", "role", "provenance"]])
display(run_systems_df[["candidate_id", "interaction", "surface_id", "role", "formula"]])


No data/custom_molecule_manifest.csv found; using the starter molecule panel.
Run systems: 6 adsorption system(s); 3 molecule reference(s).


,candidate_id,role,molecule_class,passivating_surface_id,structure_path
0,EC,baseline,carbonate,Li2CO3,data/molecules/EC.xyz
1,EMC,baseline,carbonate,Li2CO3,data/molecules/EMC.xyz
2,FEC,additive,fluorinated,LiF,data/molecules/FEC.xyz
3,VC,additive,carbonate,Li2CO3,data/molecules/VC.xyz
4,TMP,additive,phosphate,Li3PO4,data/molecules/TMP.xyz
5,succinonitrile,additive,nitrile,Li3N,data/molecules/succinonitrile.xyz


,surface_id,role,provenance
0,Li_metal,reactive,built in notebook from bcc Li bulk with pymatg...
1,LiF,passivating,built in notebook from rocksalt LiF bulk with ...
2,Li2CO3,passivating,built in notebook from COD 9008283 zabuyelite ...
3,Li3PO4,passivating,placeholder metadata; add a bulk-derived build...
4,Li3N,passivating,placeholder metadata; add a bulk-derived build...


,candidate_id,interaction,surface_id,role,structure_path
0,FEC,li_metal,Li_metal,additive,data/molecules/FEC.xyz
1,FEC,passivating,LiF,additive,data/molecules/FEC.xyz
2,VC,li_metal,Li_metal,additive,data/molecules/VC.xyz
3,VC,passivating,Li2CO3,additive,data/molecules/VC.xyz
4,EMC,li_metal,Li_metal,baseline,data/molecules/EMC.xyz
5,EMC,passivating,Li2CO3,baseline,data/molecules/EMC.xyz


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1.


In [15]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_fire2_maxstep=TOOLKIT_FIRE2_MAXSTEP,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


Native Toolkit API available: AtomicData, Batch.from_data_list, MACEWrapper, DFTD3ModelWrapper, PipelineModelWrapper, and FIRE2.
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument


/home/shadeform/ALCHEMI-Bootcamp/challenge-sei/.venv/lib/python3.12/site-packages/mace/modules/models.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "atomic_numbers", torch.tensor(atomic_numbers, dtype=torch.int64)


Toolkit relaxation engine ready: toolkit


## 3. Build And Relax The Jobs

As in Part 1, clean slabs are relaxed first. Adsorption starts are then built on the relaxed slabs as a compact `site x orientation x rotation x height` grid.


In [16]:
# Geometries are built in code (challenge_utils.molecules + custom_molecules.py);
# no structure files are read.
molecule_atoms = {
    row.candidate_id: build_molecule(row.candidate_id)
    for row in run_challenge_df.itertuples(index=False)
}
surface_meta = surfaces_df.set_index("surface_id")
used_surface_ids = sorted(run_systems_df["surface_id"].unique())

adsorption_surface_atoms = build_adsorption_surfaces(used_surface_ids, settings=SETTINGS)
display(surface_summary(adsorption_surface_atoms, settings=SETTINGS))

gas_jobs = make_gas_jobs(run_challenge_df, molecule_atoms, settings=SETTINGS)
clean_surface_jobs = make_clean_surface_jobs(
    used_surface_ids,
    adsorption_surface_atoms,
    surface_meta,
    settings=SETTINGS,
)
print(f"Gas jobs           : {len(gas_jobs)}")
print(f"Clean-surface jobs : {len(clean_surface_jobs)}")

OUTPUT_DIR.mkdir(exist_ok=True)
gas_results = relax_structures(
    gas_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_gas",
)
clean_surface_results = relax_structures(
    clean_surface_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_clean_surface",
)
require_all_converged(gas_results, label="gas references")
require_all_converged(clean_surface_results, label="clean-surface references")

combined_jobs = make_combined_jobs(
    run_systems_df,
    molecule_atoms,
    surface_meta,
    clean_surface_results,
    settings=SETTINGS,
)
print(f"Combined starts    : {len(combined_jobs)}")

all_combined_results = relax_structures(
    combined_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_combined",
)
combined_results = select_lowest_energy_site_results(all_combined_results)
require_all_converged(combined_results, label="selected combined adsorption systems")
display(selected_site_summary(combined_results))
print("Relaxations complete and converged.")


,surface_id,cell_A,atoms,mobile_surface_atoms,provenance
0,Li2CO3,14.6 x 9.9 x 29.4,144,72,zabuyelite-Li2CO3(001)-COD9008283
1,LiF,17.1 x 12.1 x 26.0,144,72,rocksalt-LiF(100)-physical
2,Li_metal,17.5 x 10.5 x 28.8,90,45,bcc-Li(100)-physical


Gas jobs           : 3
Clean-surface jobs : 3
  Cached response saved: solution_sei_gas_001
  Cached response saved: solution_sei_gas_002
  Cached response saved: solution_sei_clean_surface_001
  Cached response saved: solution_sei_clean_surface_002
Combined starts    : 24
  Cached response saved: solution_sei_combined_001
  Cached response saved: solution_sei_combined_002
  Cached response saved: solution_sei_combined_003
  Cached response saved: solution_sei_combined_004
  Cached response saved: solution_sei_combined_005
  Cached response saved: solution_sei_combined_006
  Cached response saved: solution_sei_combined_007
  Cached response saved: solution_sei_combined_008
  Cached response saved: solution_sei_combined_009
  Cached response saved: solution_sei_combined_010
  Cached response saved: solution_sei_combined_011
  Cached response saved: solution_sei_combined_012


,candidate_id,interaction,surface_id,site_label,start_orientation,azimuth_deg,energy_eV,fmax_eV_A,surface_max_displacement_A
0,EMC,li_metal,Li_metal,center,O-down,0.0,-247.931610,0.044530,0.623923
1,EMC,passivating,Li2CO3,top_C27,O-down,0.0,-1023.600708,0.046047,0.794945
2,FEC,li_metal,Li_metal,center,F-down,0.0,-227.329330,0.046430,0.846401
3,FEC,passivating,LiF,top_Li118,O-down,0.0,-750.959534,0.048082,0.154005
4,VC,li_metal,Li_metal,center,O-down,0.0,-215.162827,0.049232,0.419241
5,VC,passivating,Li2CO3,top_C75,O-down,0.0,-991.435547,0.049004,0.329110


Relaxations complete and converged.


## 4. Compute Binding Energies

Use the same adsorption-energy convention as Part 1: `E_bind = E_surface+species - E_surface - E_species`.


In [17]:
raw_component_energies_df = component_energy_table(
    run_systems_df,
    gas_results,
    clean_surface_results,
    combined_results,
)
raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
print(f"Wrote {RAW_COMPONENT_ENERGIES_PATH}")
display(raw_component_energies_df)

binding_df = binding_energy_table(run_challenge_df, raw_component_energies_df)
display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


Wrote outputs/raw_component_energies.csv


,candidate_id,interaction,surface_id,E_surface_species_eV,E_surface_eV,E_species_eV,selected_site_label,selected_start_orientation,selected_azimuth_deg
0,FEC,li_metal,Li_metal,-227.329330,-161.196426,-61.299889,center,F-down,0.0
1,FEC,passivating,LiF,-750.959534,-688.896606,-61.299889,top_Li118,O-down,0.0
2,VC,li_metal,Li_metal,-215.162827,-161.196426,-52.123089,center,O-down,0.0
3,VC,passivating,Li2CO3,-991.435547,-938.285278,-52.123089,top_C75,O-down,0.0
4,EMC,li_metal,Li_metal,-247.931610,-161.196426,-84.178146,center,O-down,0.0
5,EMC,passivating,Li2CO3,-1023.600708,-938.285278,-84.178146,top_C27,O-down,0.0


,candidate_id,role,E_bind_Li_eV,E_bind_passivating_eV
0,EMC,baseline,-2.557037,-1.137283
1,FEC,additive,-4.833015,-0.763039
2,VC,additive,-1.843311,-1.027180


## 5. Inspect Relaxed Geometries

Write the selected relaxed adsorption structures as EXTXYZ and show them with the Part 1 OVITO widget helper when available.


In [18]:
OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"
inspection_df = write_ovito_inspection_structures(combined_results, output_dir=OVITO_STRUCTURE_DIR)
INSPECT_CANDIDATE_IDS = choose_inspection_candidates(binding_df)

print(f"Wrote {len(inspection_df)} OVITO structure file(s) to {OVITO_STRUCTURE_DIR}")
print("Inspecting:", ", ".join(INSPECT_CANDIDATE_IDS))
display(inspection_df[inspection_df["candidate_id"].isin(INSPECT_CANDIDATE_IDS)])

try:
    display_widgets_grid(
        inspection_widget_rows(inspection_df, INSPECT_CANDIDATE_IDS),
        width="390px",
        height="310px",
        show_cell=False,
    )
except Exception as exc:
    print(f"OVITO widget display unavailable: {type(exc).__name__}: {exc}")
    display(inspection_df[["candidate_id", "interaction", "surface_id", "structure_path"]])


Wrote 6 OVITO structure file(s) to outputs/ovito_structures
Inspecting: EMC, FEC, VC


,candidate_id,interaction,surface_id,energy_eV,converged,structure_path
0,EMC,li_metal,Li_metal,-247.931610,True,outputs/ovito_structures/EMC_li_metal_Li_metal...
1,EMC,passivating,Li2CO3,-1023.600708,True,outputs/ovito_structures/EMC_passivating_Li2CO...
2,FEC,li_metal,Li_metal,-227.329330,True,outputs/ovito_structures/FEC_li_metal_Li_metal...
3,FEC,passivating,LiF,-750.959534,True,outputs/ovito_structures/FEC_passivating_LiF.e...
4,VC,li_metal,Li_metal,-215.162827,True,outputs/ovito_structures/VC_li_metal_Li_metal....
5,VC,passivating,Li2CO3,-991.435547,True,outputs/ovito_structures/VC_passivating_Li2CO3...


## 6. Compute Reward Scores

The challenge rubric rewards moderate Li-metal binding for SEI seeding and weak binding on the passivating proxy surface.


In [19]:
scored_df = binding_df.copy()
scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)

display(scored_df[["candidate_id", "role", "seeding_score", "passivation_score"]])


,candidate_id,role,seeding_score,passivation_score
0,EMC,baseline,0.000000,0.000000
1,FEC,additive,0.000000,0.073923
2,VC,additive,0.313377,0.000000


## 7. Pareto Front And Hypervolume

Treat both scores as objectives to maximize. Hypervolume improvement is measured against the baseline `EC`/`EMC` front with reference point `(0, 0)`.


In [20]:
final_df = scored_df.copy()
points = list(zip(final_df["seeding_score"], final_df["passivation_score"]))
final_df["is_pareto"] = pareto_flags(points)

baseline_points = list(zip(
    final_df.loc[final_df["role"].eq("baseline"), "seeding_score"],
    final_df.loc[final_df["role"].eq("baseline"), "passivation_score"],
))
baseline_hv = hypervolume_2d(baseline_points)

final_df["hypervolume_improvement"] = [
    0.0 if row.role == "baseline"
    else hypervolume_2d([*baseline_points, (row.seeding_score, row.passivation_score)]) - baseline_hv
    for row in final_df.itertuples(index=False)
]

print(f"Baseline hypervolume: {baseline_hv:.4f}")
display(final_df.sort_values("hypervolume_improvement", ascending=False))


Baseline hypervolume: 0.0000


,candidate_id,role,molecule_class,passivating_surface_id,E_bind_Li_eV,E_bind_passivating_eV,seeding_score,passivation_score,is_pareto,hypervolume_improvement
0,EMC,baseline,carbonate,Li2CO3,-2.557037,-1.137283,0.000000,0.000000,False,0.0
1,FEC,additive,fluorinated,LiF,-4.833015,-0.763039,0.000000,0.073923,True,0.0
2,VC,additive,carbonate,Li2CO3,-1.843311,-1.027180,0.313377,0.000000,True,0.0


## 8. Select Your Additive And Submit

Mark exactly one additive as selected: the additive with the largest hypervolume improvement.


In [21]:
submission = final_df.copy()
additives = submission[submission["role"].eq("additive")].copy()
if additives.empty:
    raise RuntimeError("No additive rows are available to select.")

selected_id = (
    additives
    .sort_values(["hypervolume_improvement", "candidate_id"], ascending=[False, True])
    .iloc[0]["candidate_id"]
)
submission["selected"] = submission["candidate_id"].eq(selected_id)

print(f"Selected additive: {selected_id}")
display(submission[[
    "candidate_id", "role", "seeding_score", "passivation_score",
    "is_pareto", "hypervolume_improvement", "selected",
]].sort_values("hypervolume_improvement", ascending=False))


Selected additive: FEC


,candidate_id,role,seeding_score,passivation_score,is_pareto,hypervolume_improvement,selected
0,EMC,baseline,0.000000,0.000000,False,0.0,False
1,FEC,additive,0.000000,0.073923,True,0.0,True
2,VC,additive,0.313377,0.000000,True,0.0,False


In [22]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


Wrote outputs/challenge_submission.csv


,candidate_id,role,molecule_class,passivating_surface_id,E_bind_Li_eV,E_bind_passivating_eV,seeding_score,passivation_score,is_pareto,hypervolume_improvement,selected
0,EMC,baseline,carbonate,Li2CO3,-2.557037,-1.137283,0.000000,0.000000,False,0.0,False
1,FEC,additive,fluorinated,LiF,-4.833015,-0.763039,0.000000,0.073923,True,0.0,True
2,VC,additive,carbonate,Li2CO3,-1.843311,-1.027180,0.313377,0.000000,True,0.0,False


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Leung et al., [Stability of Solid Electrolyte Interphase Components on Lithium Metal and Reactive Anode Material Surfaces](https://doi.org/10.1021/acs.jpcc.5b11719), J. Phys. Chem. C 2016; examples use large periodic SEI/Li cells and matching slab-interface references.
- Chanussot et al., [The Open Catalyst 2020 Dataset and Community Challenges](https://doi.org/10.1021/acscatal.0c04525), ACS Catalysis 2021; summarizes common slab-adsorbate setup, adsorption-energy references, vacuum, and fixed subsurface atoms.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
